<a href="https://colab.research.google.com/github/chelseanbr/aih-final-project/blob/main/notebooks/03-finetune-model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning TinyLLama with Generated MCQs

## Validate & Format Training Data

In [ ]:
def format_prompt(prompt):
    return f"""You are a helpful medical assistant.
Answer the following multiple-choice question. First, briefly explain your reasoning based on medical knowledge. Then state only one final answer using a single <answer></answer> tag (e.g., <answer>C</answer>).
Question:
{prompt}
"""

In [ ]:
import json

with open("../data/generated_mcqs.jsonl") as f:
    raw_data = [json.loads(line) for line in f]

final_mcqs_full = []
for row in raw_data:
    # Validate
    invalid_choices = "A. A skin condition\nB. A viral infection\nC. A lung disease that causes airway inflammation\nD. A type of heart failure"
    if row["mcq_choices"] == invalid_choices and row["mcq_answer"] == "C":
        continue
    # Format
    prompt = row["question"].strip() + "\n" + row["mcq_choices"].strip()
    completion = row["mcq_reason"].strip() + "\n<answer>" + row["mcq_answer"].strip() + "</answer>"
    final_mcqs_full.append({"prompt": format_prompt(prompt), "completion": completion})

# Save validated & formatted dataset
with open("../data/final_mcqs_full.jsonl", "w") as f:
    for item in final_mcqs_full:
        f.write(json.dumps(item) + "\n")

## Split Data into Train/Test

In [ ]:
import json
import random

# Load data
with open("../data/final_mcqs_full.jsonl", "r") as f:
    records = [json.loads(line) for line in f]

# Shuffle
random.seed(42)
random.shuffle(records)

# Split: 80% train, 20% test
split_idx = int(0.8 * len(records))
train_set = records[:split_idx]
test_set = records[split_idx:]

# Save to files
with open("../data/final_mcqs_train.jsonl", "w") as f:
    for entry in train_set:
        f.write(json.dumps(entry) + "\n")

with open("../data/final_mcqs_test.jsonl", "w") as f:
    for entry in test_set:
        f.write(json.dumps(entry) + "\n")

print(f"Split complete: {len(train_set)} train, {len(test_set)} test")


Split complete: 492 train, 123 test


## Supervised Fine-Tuning

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q datasets transformers peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# Get input from Google Drive
file_path = "/content/drive/MyDrive/final_mcqs_train.jsonl"

# Load full dataset
raw_dataset = load_dataset("json", data_files={"train": file_path})["train"]

# Split into 90% train, 10% validation
split_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split_dataset["train"]
val_ds = split_dataset["test"]

# Load tokenizer and model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Required for correct padding
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # load_in_8bit=True,
    torch_dtype=torch.float16,
).to(device)

# Apply LoRA with PEFT
rank = 16
lora_alpha = rank * 4
peft_config = LoraConfig(target_modules="all-linear", bias="none", task_type="CAUSAL_LM",
                         r=rank, lora_alpha=lora_alpha, lora_dropout=0.1)
model = get_peft_model(model, peft_config).to(device)
model.enable_input_require_grads()

# Tokenization
def format_and_tokenize(example):
    full_text = f"{example['prompt']}\n{example['completion']}"
    return tokenizer(full_text, truncation=True, padding="max_length", max_length=512)

tokenized_train = train_ds.map(format_and_tokenize).remove_columns(["prompt", "completion"])
tokenized_val = val_ds.map(format_and_tokenize).remove_columns(["prompt", "completion"])

# Google Drive output paths
output_dir = "/content/drive/MyDrive/tinyllama-mcq-lora"
logging_dir = "/content/drive/MyDrive/logs"

# Training config
training_args = TrainingArguments(
    gradient_checkpointing=True,
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    weight_decay=0.01,
    fp16=True,
    learning_rate=2e-4,
    logging_dir=logging_dir,
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

Using device: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
50,1.009900
100,0.896600
150,0.739000
200,0.613500
250,0.586800
300,0.421900
350,0.420700


('/content/drive/MyDrive/tinyllama-mcq-lora/tokenizer_config.json',
 '/content/drive/MyDrive/tinyllama-mcq-lora/special_tokens_map.json',
 '/content/drive/MyDrive/tinyllama-mcq-lora/tokenizer.model',
 '/content/drive/MyDrive/tinyllama-mcq-lora/added_tokens.json',
 '/content/drive/MyDrive/tinyllama-mcq-lora/tokenizer.json')

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# === Load dataset from Google Drive ===
file_path = "/content/drive/MyDrive/final_mcqs_train.jsonl"
raw_dataset = load_dataset("json", data_files={"train": file_path})["train"]

# === Split into train/val ===
split_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split_dataset["train"]
val_ds = split_dataset["test"]

# === Load tokenizer and model ===
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
).to(device)

# === Apply LoRA ===
rank = 16
lora_alpha = rank * 4
peft_config = LoraConfig(
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
    r=rank,
    lora_alpha=lora_alpha,
    lora_dropout=0.1
)
model = get_peft_model(model, peft_config).to(device)
model.enable_input_require_grads()

# === Tokenization function ===
def format_and_tokenize(example):
    full_text = f"{example['prompt']}\n{example['completion']}"
    return tokenizer(full_text, truncation=True, padding="max_length", max_length=512)

tokenized_train = train_ds.map(format_and_tokenize).remove_columns(["prompt", "completion"])
tokenized_val = val_ds.map(format_and_tokenize).remove_columns(["prompt", "completion"])

# === Google Drive paths ===
output_dir = "/content/drive/MyDrive/tinyllama-mcq-lora"
logging_dir = "/content/drive/MyDrive/logs"

# === Training arguments ===
training_args = TrainingArguments(
    output_dir=output_dir,
    logging_dir=logging_dir,
    gradient_checkpointing=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    weight_decay=0.01,
    fp16=True,
    learning_rate=2e-4,
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none"
)

# === Trainer setup ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

# === Save best model ===
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

## Quick Eval

In [ ]:
from datasets import load_dataset
import json

# Load your held-out test set
test_file = "/content/drive/MyDrive/final_mcqs_test.jsonl"
test_data = [json.loads(line) for line in open(test_file)]


In [ ]:
import re
from tqdm import tqdm
import torch

def evaluate_model_on_subset(model, tokenizer, data, subset_size=50):
    model.eval()
    correct = 0
    total = 0

    # Track answer letter distribution
    num_a = 0
    num_b = 0
    num_c = 0
    num_d = 0
    num_skipped = 0

    for row in tqdm(data[:subset_size]):
        prompt = row["prompt"].strip()
        true_answer = re.search(r"<answer>([A-D])</answer>", row["completion"])
        true_answer = true_answer.group(1) if true_answer else None

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated = decoded[len(prompt):].strip()

        answers = re.findall(r"<answer>([A-D])</answer>", generated)

        if len(answers) != 1:
            num_skipped += 1
            continue  # Skip samples with no or multiple answers

        pred_answer = answers[0]

        # Count answer distribution
        if pred_answer == "A":
            num_a += 1
        elif pred_answer == "B":
            num_b += 1
        elif pred_answer == "C":
            num_c += 1
        elif pred_answer == "D":
            num_d += 1

        if pred_answer == true_answer:
            correct += 1

        total += 1

    accuracy = correct / total if total > 0 else 0

    print(f"\nAccuracy on {total} valid samples: {accuracy:.2%} ({correct}/{total})")
    print(f"Skipped due to multiple/no <answer>: {num_skipped}")
    print("Answer Distribution:")
    print(f"  A: {num_a}")
    print(f"  B: {num_b}")
    print(f"  C: {num_c}")
    print(f"  D: {num_d}")


In [ ]:
evaluate_model_on_subset(trainer.model, tokenizer, test_data, subset_size=5)